In [1]:
import torch.nn as nn
from girth import grm_mml
from torch.utils.data import dataset


In [2]:
import torch
from torch import nn
from torch.nn import functional as F


class VSOSpred(nn.Module):
    def __init__(self, num_usrs: int, num_olimps: int):
        super().__init__()
        self.global_strong = nn.Embedding(
            num_usrs,
            1,
        )
        self.performance = nn.Embedding(
            num_usrs * num_olimps,
            1,
        )
        nn.init.zeros_(self.global_strong.weight)
        nn.init.zeros_(self.performance.weight)
        self.num_olimps = num_olimps
    def get_strong(self, usrs_ids, olimps_ids) -> torch.Tensor:
        global_strength = self.global_strong(usrs_ids).squeeze(-1)
        joint_ids = (
            usrs_ids * self.num_olimps
            + olimps_ids
        )
        local = self.performance(
            joint_ids
        ).squeeze(-1)
        return global_strength + local
    def forward(self, usr_a_ids, usr_b_ids, olimp_ids):
        strength_a = self.get_strong(
            usr_a_ids,
            olimp_ids,
        )
        strength_b = self.get_strong(
            usr_b_ids,
            olimp_ids,
        )
        logits = strength_a - strength_b
        return {
            "logits": logits,
            "strength_a": strength_a,
            "strength_b": strength_b,
        }
    def regularization_loss(self, global_weight, offset_weight):
        global_penalty = (
            self.global_strong.weight.pow(2).mean()
        )
        offset_penalty = (
            self.performance.weight.pow(2).mean()
        )
        return (
            global_weight * global_penalty
            + offset_weight * offset_penalty
        )

In [3]:
from torch.utils.data import Dataset
import numpy as np
import pandas as pd
from pathlib import Path


class VSOSet(Dataset):
    def __init__(self, directory):
        directory = Path(directory)
        self.data = []
        d1 = pd.read_csv(directory / "belchonok.csv")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]



KeyError: 'name'

In [14]:
import pandas as pd
directory = '/home/pirojban/hahaha/LMSH_AI_HACK/src/Vlad/preprocessed_data'
d1 = pd.read_csv(directory + "/belchonok.csv")
d2 = pd.read_csv(directory + "/fin.csv")
d3 = pd.concat([d1, d2])
d3

,name,total,result,year,id,region,class_study,p1,p2,p3,p4,p5,p6,p7,p8,score,title
0,Хакимзянов И.,51.0,3.0,24-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Хмелевский М.,48.0,3.0,24-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Еникеев К.,45.0,3.0,24-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Главатских В.,41.0,3.0,24-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Савицкая Т.,40.0,3.0,24-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,Зуева А.,NaN,NaN,25-26,482.0,Камчатский край,11.0,100.0,11.0,13.0,0.0,0.0,15.0,0.0,6.0,145.0,NaN
975,Кочуров Е.,NaN,NaN,25-26,483.0,Кировская область,10.0,42.0,9.0,20.0,13.0,36.0,0.0,10.0,0.0,130.0,NaN
976,Голов С.,NaN,NaN,25-26,484.0,Республика Мордовия,11.0,100.0,9.0,0.0,0.0,0.0,15.0,0.0,0.0,124.0,NaN
977,Курзин И.,NaN,NaN,25-26,485.0,Ульяновская область,9.0,36.0,9.0,13.0,0.0,36.0,10.0,10.0,6.0,120.0,NaN


In [ ]:
num_usrs = 500

In [ ]:
model = VSOSpred(
    num_usrs=num_usrs,
    num_olimps=7,
)
optim = torch.optim.AdamW(model.parameters())
for batch in dataloader:
    out = model(
        athlete_a_ids=batch["athlete_a"],
        athlete_b_ids=batch["athlete_b"],
        competition_ids=batch["competition_id"],
    )
    pairwise_loss = F.binary_cross_entropy_with_logits(
        out["logits"],
        batch["target"].float(),
    )
    loss = (
        pairwise_loss
        + model.regularization_loss(1e-3, 1e-1)
    )
    loss.backward()
    optim.step()
    optim.zero_grad()